# Figure 1: genome-scale Perturb-seq (Replogle et al.)

Run this notebook from the `figures/` folder after populating `data/` and `results/`
(see the repository README). Each section below reruns the relevant part of an analysis
notebook from the saved models; panels are written to `figures/panels/`
(`paths.FIGURES_DIR`).

Panels a–c are schematics.

In [ ]:
import os
import sys

# Repository root, so that paths.py (DATA_DIR / RESULTS_DIR / FIGURES_DIR) can be imported.
sys.path.insert(0, os.path.abspath('..'))
import paths

for d in (paths.DATA_DIR, paths.RESULTS_DIR, paths.FIGURES_DIR):
    print(d, '(found)' if os.path.isdir(d) else '(missing)')

## Replogle: perturbation groups, pathway recovery and enrichment

From [`scripts/replogle/1_replogle_analysis.ipynb`](../scripts/replogle/1_replogle_analysis.ipynb).

The enrichment cell queries Enrichr through `gseapy` and needs internet access.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

import seaborn as sns
import matplotlib.pyplot as plt

import torch

import sherlock as slk


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'non-targeting')
print('\n')
slk.configs.show()
slk.configs.reset_config()
slk.configs.get_config('ntc_label')


In [ ]:
data = slk.datasets.replogle()

'''Alternatively for your own data:
data = sc.read_h5ad(PATH_TO_YOUR_DATA)
slk_prepare_data(data, pert_key=PERT_KEY, ntc_label=NTC_LABEL, treatment_key=TREATMENT_KEY|None, inplace=True)
'''


In [ ]:
# data (all cells, all perts)
#  ├── sub            → remove NTC, UMAP on Sherlock z
#  │    ├── z_sub     → keep only clustered genes, use sub's UMAP
#  │    └── z_sub_umap→ keep only clustered genes, recompute UMAP on expression
#  └── data_sub       → keep clustered genes + NTC, for DEG analysis


In [ ]:
perturbations = set(np.unique(data.obs[slk.configs.get_config('pert_key')].values))
print("Perturbations:", len(perturbations))

pathways = []
for pathway in data.uns['pathways'].values():
    pathways.append(set(pathway))
sum([len(pathways[i]) for i in range(len(pathways))])
print("Pathways:", len(pathways))

overlap = any(len(set(pathways[i]) & set(pathways[j])) > 0 for i in range(len(pathways)) for j in range(i + 1, len(pathways)))
print("Overlap exists between pathways:" if overlap else "No overlap between pathways.")


In [ ]:
slk.pp.treat_effect(
    data,
    label_col = 'pert',
    control_label = 'NTC',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)


In [ ]:
NUM_EPOCHS     = 1000
VALIDATE_EVERY = 50
BATCH_SIZE     = 4096
LATENT_DIM     = 16
PATIENCE       = 20

In [ ]:
SHERLOCK_KWARGS = {
    # "tau_end": 0.3,
    # # "tau_init": 1.5,
    # "cov_lambda": 1e-5,
    # "H_lambda": 1e-1,
    # "gate_row_repulsion_lambda": 1e-2,
    # "ce_lambda": 10,
    # "shift": "linear",
    "l0_lambda": 10,
    "debug": True,
    # "lr": 1e-3,
    # "ntc_lambda": 1e-5,       
    # "pert_lambda": 1,
    "n_epochs_l0_warmup": 200
}

In [ ]:
seed = 1

In [ ]:
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run(
#         data,
#         model="vae",
#         batch_size=BATCH_SIZE,
#         num_epochs=NUM_EPOCHS,
#         device=DEVICE,
#         latent_dim=LATENT_DIM,
#         patience=PATIENCE,
#         validate_every=VALIDATE_EVERY,
#         seed=seed,
#         # debug=True,
#         **SHERLOCK_KWARGS,
#     )

In [ ]:
#results

In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/replogle_model_{seed}.pth')

# print(f'Saved results to ../data/replogle_model_{seed}.pth')

In [ ]:
#model = results['model']

In [ ]:
model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/replogle_model_1.pth')


In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")


In [ ]:
data.uns['results']


In [ ]:
slk.pl.plot_r2(data)


In [ ]:
slk.pl.plot_ev(data, cmax=2, top_n=50)


In [ ]:
slk.tl.ev_sig(data, cond_idx=0)


In [ ]:
# Useful dataframe that plot_ev_sig uses internally
slk.pl.make_ev_long_df(data, cond_idx=0, uns_key="results")


In [ ]:
slk.pl.plot_ev_sig(
    data,
    cond_idx=0,
    uns_key="results",
    alpha=0.05,
    top_n_genes=10,
    min_sig_genes=3,
    cluster_rows=True,
    cluster_cols=False,
    use_clustermap=True,
)

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z', random_state=0)
sc.tl.umap(sub, random_state=0)

import random
n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.shuffle(palette)
sc.pl.umap(sub, color='pert', palette=palette, frameon=False)

In [ ]:
zrc = data.uns['results']['z_rho_corr']

print(f"Spearman ρ = {zrc['spearman_r']:.3f}  (p = {zrc['spearman_p']:.1e})")
print(f" Pearson  r = {zrc['pearson_r']:.3f}  (p = {zrc['pearson_p']:.1e})") 

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
W = data.uns['results']['W']
plt.hist(W.flatten(), bins=30)
plt.show()


In [ ]:
W_bin = (W > 0.5).astype(float)
sns.heatmap(W, cmap="coolwarm")

In [ ]:
C = data.uns['results']['rho_corr']

#Specific subsetting for replogle. Otherwise just use C directly
all_genes = [gene for genes in data.uns['pathways'].values() for gene in genes]
dataset = slk.tl.PerturbMatchingDataset(data)
gene_ids = [dataset.perturbation_dict[g] for g in all_genes]
cov_subset = C[np.ix_(gene_ids, gene_ids)].clip(-1, 1)

In [ ]:
groups = slk.tl.cluster_rho(data, t=1.5, rho_corr=cov_subset, show=True)

In [ ]:
# Save the cluster_rho figure
# Since the function shows the plot, we need to recreate it with save capability

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

C = cov_subset
D = 1.0 - C
Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')

# Create figure
t = 1.5
plt.figure(figsize=(10, 6))
dendrogram(
    Z_link,
    color_threshold=t,
    above_threshold_color="gray",
    labels=np.arange(len(C)),
    leaf_rotation=90,
    leaf_font_size=8,
)
plt.axhline(y=t, color="red", linestyle="--", linewidth=1)
plt.title(f"Covariance hierarchal clustering t={t}")
plt.xlabel("Item")
plt.ylabel("Distance")
plt.tight_layout()

# Save the figure
plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram.pdf', bbox_inches='tight')
plt.show()

print("Figure saved to ../results/cluster_rho_dendrogram.png and .pdf")

# Get cluster labels
groups = fcluster(Z_link, t=t, criterion='distance')

In [ ]:
#for a non subsetted data, pass in genes corresponding to idx ie dataset.perturbation_dict.keys()
named_groups = slk.tl.group2name(groups, all_genes)
named_groups

In [ ]:
# map perturbation -> group using named_groups (index=gene, column='group')
gene_to_group = named_groups['group'].to_dict()

# assign group to obs based on slk pert key
sub.obs['pert_group'] = sub.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
sub.obs['pert_group'] = sub.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(sub.obs['pert_group'].value_counts().sort_index())

sc.pl.umap(sub, color=['pert_group'], title='Perturbation Groups', palette='tab20', size=10)


## From here on out it is validation of replogle

In [ ]:
from collections import Counter

# Flatten all genes from all pathways
#all_genes = [gene for genes in data.uns['pathways'].values() for gene in genes]

# Count how many times each gene appears
gene_counts = Counter(all_genes)

# Find genes that appear in more than one pathway
shared_genes = [gene for gene, count in gene_counts.items() if count > 1]

if shared_genes:
    print(f"⚠️ {len(shared_genes)} genes are shared across pathways:")
    print(shared_genes[:20])  # show first 20 for inspection
else:
    print("✅ Each gene appears in exactly one pathway.")


#pert df
pathway_genes = np.concatenate([i for i in list(data.uns['pathways'].values())])
pathway_genes = pathway_genes[pathway_genes != 'LSM5']
print(len(pathway_genes))
print(len(all_genes))
perts = pd.DataFrame(groups, index=all_genes, columns=['cluster'])
perts = perts.loc[pathway_genes]

#pathway df
gene_to_pathway = {
    gene: pathway
    for pathway, genes in data.uns['pathways'].items()
    for gene in genes if gene != 'LSM5'
}
pathway_df = pd.DataFrame.from_dict(gene_to_pathway, orient="index", columns=["cluster"])


perts = perts.drop(shared_genes, errors="ignore")
pathway_df = pathway_df.drop(shared_genes, errors="ignore")

print("After removal:")
print("perts size     :", perts.shape)
print("pathway_df size:", pathway_df.shape)


In [ ]:
pathway_df['gene'] = pathway_df.index.copy()
pathway_df.columns = ['pathway', 'gene']
pathway_df


In [ ]:
# Assign pathway to groups

pathway_df['group'] = named_groups['group']


In [ ]:
# Create table of pathway and group with counts
pathway_group_table = pathway_df.groupby(['pathway', 'group']).size().reset_index(name='count')
pathway_group_table = pathway_group_table.sort_values('group')
pathway_group_table


In [ ]:
# # Re-plot dendrogram with group labels
# from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

# C = cov_subset
# D = 1.0 - C
# Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')

# # Get groups
# t = 1.5
# groups_labeled = fcluster(Z_link, t=t, criterion='distance')

# # Create figure
# plt.figure(figsize=(12, 7))
# dend = dendrogram(
#     Z_link,
#     color_threshold=t,
#     above_threshold_color="gray",
#     labels=np.arange(len(C)),
#     leaf_rotation=90,
#     leaf_font_size=8,
#     no_plot=False
# )

# # Add group labels at the bottom
# leaf_order = dend['leaves']
# group_labels_ordered = [groups_labeled[i] for i in leaf_order]

# # Add colored rectangles for each group
# from matplotlib.patches import Rectangle
# unique_groups = sorted(set(groups_labeled))
# colors = plt.cm.tab20(np.linspace(0, 1, len(unique_groups)))
# group_to_color = {g: colors[i] for i, g in enumerate(unique_groups)}

# # Add group labels as text below x-axis
# ax = plt.gca()
# y_min = ax.get_ylim()[0]
# for i, (leaf_idx, group) in enumerate(zip(leaf_order, group_labels_ordered)):
#     ax.text(i*10 + 5, y_min - 0.15, f'G{group}', 
#             ha='center', va='top', fontsize=7, 
#             color=group_to_color[group], fontweight='bold')

# plt.axhline(y=t, color="red", linestyle="--", linewidth=1, label=f'cut threshold t={t}')
# plt.title(f"Covariance hierarchical clustering (t={t}) with group labels")
# plt.xlabel("Gene index (colored by group)")
# plt.ylabel("Distance")
# plt.legend()
# plt.tight_layout()
# plt.show()

# print(f"\nNumber of groups: {len(unique_groups)}")
# print(f"Group sizes: {pd.Series(groups_labeled).value_counts().sort_index()}")


In [ ]:
# Assign each pathway to the group with maximum count
pathway_max_group = pathway_group_table.loc[pathway_group_table.groupby('pathway')['count'].idxmax()]
pathway_max_group = pathway_max_group[['pathway', 'group', 'count']].sort_values('group')
pathway_max_group


In [ ]:
# Align on the same gene index
perts_aligned = perts.loc[pathway_df.index]

perts_aligned['gene'] = perts_aligned.index.copy()
perts_aligned


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
from scipy.optimize import linear_sum_assignment
from scipy.stats import hypergeom

try:
    from statsmodels.stats.multitest import multipletests
    _HAS_SM = True
except Exception:
    _HAS_SM = False


def align_and_summarize(
    clusters_df: pd.DataFrame,
    pathways_df: pd.DataFrame,
    gene_col: str = "gene",
    cluster_col: str = "cluster",
    pathway_col: str = "pathway",
    min_pathway_size: int = 1,
):
    # 1) Merge on gene
    df = pd.merge(
        clusters_df[[gene_col, cluster_col]],
        pathways_df[[gene_col, pathway_col]],
        on=gene_col,
        how="inner",
        validate="one_to_one"
    ).dropna(subset=[cluster_col, pathway_col]).copy()

    # Optional: filter tiny pathways to stabilize Hungarian/enrichment
    if min_pathway_size > 1:
        keep_path = df[pathway_col].value_counts()
        keep_path = keep_path[keep_path >= min_pathway_size].index
        df = df[df[pathway_col].isin(keep_path)].copy()

    # Cast to string for safe indexing
    y_true = df[pathway_col].astype(str).values
    y_pred = df[cluster_col].astype(str).values
    true_labels = np.unique(y_true)
    pred_labels = np.unique(y_pred)

    # 2) Contingency (rows=pathways, cols=clusters)
    C = pd.crosstab(df[pathway_col].astype(str), df[cluster_col].astype(str))
    contingency_df = C.reindex(index=true_labels, columns=pred_labels, fill_value=0)

    # 3) Hungarian alignment to maximize correct assignments (sum of diagonal)
    M = contingency_df.values
    row_ind, col_ind = linear_sum_assignment(M.max() - M)  # maximize
    mapping = {pred_labels[j]: true_labels[i] for i, j in zip(row_ind, col_ind)}

    # 4) Remap clusters → pathways and compute metrics
    mapped = np.array([mapping.get(cl, None) for cl in y_pred], dtype=object)
    is_match = mapped == y_true
    accuracy = float(is_match.mean())

    # Label-based external clustering metrics (label strings are fine)
    ari = adjusted_rand_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)

    # 5) Per-cluster purity & mapped target
    purity_rows = []
    for cl, g in df.groupby(cluster_col):
        majority = g[pathway_col].mode().iat[0]
        purity = (g[pathway_col] == majority).mean()
        purity_rows.append({
            "cluster": str(cl),
            "size": len(g),
            "mapped_pathway": mapping.get(str(cl), None),
            "purity": purity
        })
    purity_df = pd.DataFrame(purity_rows).sort_values("cluster").set_index("cluster")

    # 6) Normalized contingency for plotting
    contingency = contingency_df.copy()
    contingency_row_norm = contingency.div(contingency.sum(axis=1).replace(0, np.nan), axis=0)
    contingency_col_norm = contingency.div(contingency.sum(axis=0).replace(0, np.nan), axis=1)

    # 7) Pathway↔cluster enrichment (hypergeometric with FDR)
    #    N = total genes, K = pathway size, n = cluster size, k = overlap
    N = len(df)
    path_sizes = contingency.sum(axis=1)
    cl_sizes = contingency.sum(axis=0)
    records = []
    for p in contingency.index:
        K = int(path_sizes.loc[p])
        for c in contingency.columns:
            n = int(cl_sizes.loc[c])
            k = int(contingency.loc[p, c])
            # p-value: probability of ≥k overlap by chance
            pval = hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
            records.append({"pathway": p, "cluster": c, "k": k, "K": K, "n": n, "N": N, "pval": float(pval)})

    enrich_df = pd.DataFrame(records)
    if _HAS_SM and len(enrich_df) > 0:
        enrich_df["qval"] = multipletests(enrich_df["pval"].values, method="fdr_bh")[1]
    else:
        # Bonferroni fallback
        m = max(len(enrich_df), 1)
        enrich_df["qval"] = np.minimum(enrich_df["pval"] * m, 1.0)

    # Signed/visual score: -log10(q)
    with np.errstate(divide="ignore"):
        enrich_df["neglog10q"] = -np.log10(np.clip(enrich_df["qval"].values, 1e-300, 1.0))

    # 8) Attach mapped labels into the merged table
    df["mapped_pathway"] = mapped
    df["is_match"] = is_match

    return {
        "merged": df[[gene_col, cluster_col, pathway_col, "mapped_pathway", "is_match"]].rename(
            columns={gene_col: "gene", cluster_col: "cluster", pathway_col: "pathway"}
        ),
        "mapping": mapping,
        "accuracy": accuracy,
        "ari": ari,
        "ami": ami,
        "purity_df": purity_df,
        "contingency": contingency,
        "contingency_row_norm": contingency_row_norm,
        "contingency_col_norm": contingency_col_norm,
        "enrichment": enrich_df,
    }


In [ ]:
results = align_and_summarize(
    clusters_df=perts_aligned,          # gene↦cluster from covariance HC
    pathways_df=pathway_df,             # gene↦pathway ground truth
    gene_col="gene",
    cluster_col="cluster",
    pathway_col="pathway",
    min_pathway_size=1,                 # optional: filter tiny pathways
)

print("Best cluster→pathway mapping:")
for cl, pw in results["mapping"].items():
    print(f"  {cl} → {pw}")

print(f"\nGlobal alignment accuracy: {results['accuracy']:.3f}")
print(f"ARI: {results['ari']:.3f} | AMI: {results['ami']:.3f}\n")

print("Per-cluster purity:")
print(results["purity_df"])


In [ ]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_hex
import matplotlib.pyplot as plt

# --- your existing setup ---
results_df = results['merged']
clustered_genes = results_df["gene"].unique()

z_sub = sub[sub.obs["pert"].isin(clustered_genes)].copy()

gene_to_true   = dict(zip(results_df["gene"], results_df["pathway"]))
gene_to_mapped = dict(zip(results_df["gene"], results_df["mapped_pathway"]))

z_sub.obs["true_pathway"]   = z_sub.obs["pert"].map(gene_to_true)
z_sub.obs["mapped_pathway"] = z_sub.obs["pert"].map(gene_to_mapped)

# Drop NA mapped rows
z_sub = z_sub[z_sub.obs["mapped_pathway"].notna() & (z_sub.obs["mapped_pathway"] != "NA")].copy()

# --- NEW: enforce identical palettes for both columns ---
# 1) Make both columns categorical with the SAME category set & order
true_vals   = pd.Index(z_sub.obs["true_pathway"].dropna().unique().astype(str))
mapped_vals = pd.Index(z_sub.obs["mapped_pathway"].dropna().unique().astype(str))
all_cats = pd.Index(np.unique(true_vals.union(mapped_vals)))  # sorted union

cat_dtype = pd.api.types.CategoricalDtype(categories=all_cats, ordered=False)
z_sub.obs["true_pathway"]   = z_sub.obs["true_pathway"].astype("string").astype(cat_dtype)
z_sub.obs["mapped_pathway"] = z_sub.obs["mapped_pathway"].astype("string").astype(cat_dtype)

# 2) Build a single color palette over all categories
cmap = plt.cm.get_cmap("tab20", len(all_cats))  # any qualitative map works
shared_palette = {cat: to_hex(cmap(i)) for i, cat in enumerate(all_cats)}

# 3) Write per-column color lists in each column's category order
true_order   = list(z_sub.obs["true_pathway"].cat.categories)
mapped_order = list(z_sub.obs["mapped_pathway"].cat.categories)

z_sub.uns["true_pathway_colors"]   = [shared_palette[c] for c in true_order]
z_sub.uns["mapped_pathway_colors"] = [shared_palette[c] for c in mapped_order]

# --- Plot (colors now matched across panels) ---
import scanpy as sc
sc.pl.umap(
    z_sub,
    color=["true_pathway", "mapped_pathway"],
    wspace=0.4,
    save='_pathway_comparison.png'  # This will save to figures/umap_pathway_comparison.png
)

# Alternative: Save manually with more control
fig = sc.pl.umap(
    z_sub,
    color=["true_pathway", "mapped_pathway"],
    wspace=0.4,
    show=False,
    return_fig=True
)
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison.pdf', bbox_inches='tight')
plt.show()

print("Figure saved to ../results/umap_pathway_comparison.png and .pdf")


In [ ]:
# # Save the cluster_rho figure — leaf labels colored by cluster -> pathway -> shared_palette

# from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

# C = cov_subset
# D = 1.0 - C
# Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')

# t = 1.5
# groups = fcluster(Z_link, t=t, criterion='distance')

# # Build cluster -> pathway mapping from purity_df
# # purity_df index is cluster (string), mapped_pathway column
# cluster_to_pathway = results["purity_df"]["mapped_pathway"].to_dict()  # {'1': 'MEDIATOR_COMPLEX', ...}

# # pert index -> color via cluster -> pathway -> shared_palette
# fallback = 'black'
# leaf_colors = [
#     shared_palette.get(str(cluster_to_pathway.get(str(groups[i]), '')), fallback)
#     for i in range(len(groups))
# ]

# # Branch coloring: color branch if all leaves underneath share the same pathway
# n = len(groups)
# node_pathways = {i: {str(cluster_to_pathway.get(str(groups[i]), ''))} for i in range(n)}
# for k, (left, right, _, _) in enumerate(Z_link):
#     node_id = n + k
#     node_pathways[node_id] = node_pathways[int(left)] | node_pathways[int(right)]

# def link_color_func(k):
#     paths = node_pathways[k]
#     if len(paths) == 1:
#         return shared_palette.get(next(iter(paths)), 'gray')
#     return 'gray'

# plt.figure(figsize=(5, 3))
# dendrogram(
#     Z_link,
#     color_threshold=0,
#     above_threshold_color="gray",
#     labels=np.arange(len(C)),
#     leaf_rotation=90,
#     leaf_font_size=2,
#     link_color_func=link_color_func,
# )
# plt.axhline(y=t, color="red", linestyle="--", linewidth=1)
# plt.title(f"Covariance hierarchical clustering t={t}")
# plt.xlabel("Item")
# plt.ylabel("Distance")

# # Color leaf labels
# ax = plt.gca()
# for lbl, color in zip(ax.get_xticklabels(), [leaf_colors[i] for i in dendrogram(
#         Z_link, no_plot=True)["leaves"]]):
#     lbl.set_color(color)

# plt.tight_layout()
# plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram.png', dpi=300, bbox_inches='tight')
# plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram.pdf', bbox_inches='tight')
# plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram.svg', bbox_inches='tight')
# plt.show()
# print("Figure saved.")


In [ ]:
# # Dendrogram with leaf labels / branches colored by TRUE pathway
# # Uses all_genes (same order as cov_subset rows) and gene_to_true

# from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

# C = cov_subset
# D = 1.0 - C
# Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
# t = 1.5

# # Map each leaf index -> gene -> true_pathway -> color
# fallback = '#cccccc'
# leaf_true_colors = [
#     shared_palette.get(str(gene_to_true.get(all_genes[i], '')), fallback)
#     for i in range(len(all_genes))
# ]

# # Color branches: gray unless all leaves underneath share the same true pathway
# n = len(all_genes)
# node_pathways = {i: {str(gene_to_true.get(all_genes[i], ''))} for i in range(n)}
# for k, (left, right, _, _) in enumerate(Z_link):
#     node_id = n + k
#     node_pathways[node_id] = node_pathways[int(left)] | node_pathways[int(right)]

# def link_color_func_true(k):
#     paths = node_pathways[k]
#     if len(paths) == 1:
#         return shared_palette.get(next(iter(paths)), 'gray')
#     return 'gray'

# fig, ax = plt.subplots(figsize=(5, 3))
# dend = dendrogram(
#     Z_link,
#     color_threshold=0,
#     above_threshold_color='gray',
#     labels=np.arange(len(C)),
#     leaf_rotation=90,
#     leaf_font_size=2,
#     link_color_func=link_color_func_true,
#     ax=ax,
# )
# ax.axhline(y=t, color='red', linestyle='--', linewidth=1)
# ax.set_xlabel('Perturbation')
# ax.set_ylabel('Distance')
# ax.set_title('Hierarchical clustering — colored by true pathway')

# # Color leaf labels by true pathway
# leaf_order = dend['leaves']
# for lbl, idx in zip(ax.get_xticklabels(), leaf_order):
#     lbl.set_color(leaf_true_colors[idx])

# # Legend
# from matplotlib.patches import Patch
# shown = sorted({str(gene_to_true.get(g, '')) for g in all_genes} - {''})
# legend_elements = [Patch(facecolor=shared_palette.get(p, fallback), label=p) for p in shown]
# fig.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.0, 0.5),
#            title='True pathway', fontsize=8, title_fontsize=9, frameon=False)

# plt.tight_layout()
# plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram_true_pathway.png', dpi=300, bbox_inches='tight')
# plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram_true_pathway.pdf', dpi=300, bbox_inches='tight')
# plt.savefig(f'{paths.FIGURES_DIR}/replogle/cluster_rho_dendrogram_true_pathway.svg', dpi=300, bbox_inches='tight')
# plt.show()
# print("Saved to ../results/cluster_rho_dendrogram_true_pathway.[png|pdf|svg]")


In [ ]:
ev_df = slk.pl.make_ev_long_df(data)


merged = results["merged"].copy()

path_map = merged[["gene", "pathway", "mapped_pathway", "is_match"]].rename(
    columns={"gene": "pert"}
)

ev_annot = ev_df.merge(path_map, on="pert", how="inner")

ev_annot["pathway_used"] = ev_annot["mapped_pathway"].astype(str)

alpha = 0.05
ev_sig = ev_annot[ev_annot["qval"] < alpha].copy()
if ev_sig.empty:
    print(f"No significant (pert, gene) pairs at q < {alpha}.")
else:
    gp = ev_sig.groupby(["pathway_used", "gene"])
    pathway_targets = (
        gp.agg(
            n_perts=("pert", "nunique"),
            mean_ev=("ev", "mean"),
            mean_qval=("qval", "mean"),
        )
        .reset_index()
        .rename(columns={"pathway_used": "pathway", "gene": "target_gene"})
        .sort_values(["pathway", "mean_qval", "target_gene"])
    )
    print(
        pathway_targets
    )


In [ ]:
#! pip install gseapy


In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib as mpl
# import gseapy as gp
# import time
# mpl.rcParams.update({
#     'font.size': 10,
#     'svg.fonttype': 'none',
#     'font.family': 'Arial',
# })
# mpl.rcParams['axes.titlesize'] = 10
# mpl.rcParams['xtick.labelsize'] = 10  # Colorbar tick label size 
# mpl.rcParams['ytick.labelsize'] = 10 # Colorbar tick label size

# def run_group_enrichment_hallmark(
#     pathway_targets: pd.DataFrame,
#     min_genes_per_group: int = 5,
#     max_genes_per_group: int = 200,
#     hallmark_lib: str = "MSigDB_Oncogenic_Signatures",  # C6 here
#     organism: str = "Human",
#     top_terms_per_group: int = 5,
#     adj_alpha: float = 0.05,   # <-- significance cutoff on Adjusted P-value
# ):
#     """
#     For each internal group/pathway, run enrichment on its target genes
#     and return a tidy table of *significant* enriched terms.

#     Expected columns in pathway_targets:
#       - 'pathway'     : internal label (EXOSOME, SPLICEOSOME, ...)
#       - 'target_gene' : gene symbol
#       - (optional) 'mean_ev' : used to sort genes before truncating
#     """
#     df = pathway_targets.copy()

#     # Drop 'None' groups if present
#     df = df[df["pathway"] != "None"].copy()

#     # Sort within each pathway by mean_ev if present (strongest targets first)
#     if "mean_ev" in df.columns:
#         df = df.sort_values(["pathway", "mean_ev"], ascending=[True, False])

#     # Collect gene lists per pathway
#     path_to_genes = (
#         df.groupby("pathway")["target_gene"]
#           .apply(lambda s: list(dict.fromkeys(s.astype(str))))  # drop dups, keep order
#           .to_dict()
#     )

#     rows = []

#     for pw, genes in path_to_genes.items():
#         # Cap number of genes (to avoid huge lists)
#         genes = genes[:max_genes_per_group]

#         if len(genes) < min_genes_per_group:
#             continue

#         print(f"Running enrichment for: {pw} ({len(genes)} genes)")
#         enr = gp.enrichr(
#             gene_list=genes,
#             gene_sets=[hallmark_lib],
#             organism=organism,
#             outdir=None,
#             cutoff=1.0,  # keep all; we'll filter ourselves
#         )
#         time.sleep(1)  # avoid Enrichr 429 rate limit

#         res = enr.results.copy()
#         if res.empty:
#             continue

#         res["pathway"] = pw

#         # Parse overlap "k/n" -> k
#         def parse_k(s):
#             try:
#                 k, n = s.split("/")
#                 return int(k)
#             except Exception:
#                 return np.nan

#         res["overlap_k"] = res["Overlap"].astype(str).apply(parse_k)
#         res["minus_log10_adjP"] = -np.log10(
#             res["Adjusted P-value"].clip(lower=1e-300)
#         )

#         # === keep only significant terms ===
#         res = res[res["Adjusted P-value"] <= adj_alpha]
#         if res.empty:
#             continue

#         # Take top N significant terms by adjusted p-value
#         res = res.sort_values("Adjusted P-value").head(top_terms_per_group)

#         rows.append(
#             res[
#                 ["pathway", "Term", "Adjusted P-value",
#                  "Overlap", "overlap_k", "minus_log10_adjP"]
#             ]
#         )

#     if not rows:
#         raise ValueError("No significant enrichment results at adj_alpha "
#                          f"= {adj_alpha}. Try relaxing the cutoff or min_genes_per_group.")

#     enrich_df = pd.concat(rows, ignore_index=True)
#     return enrich_df


# def plot_hallmark_dotplot(enrich_df: pd.DataFrame, title: str = ""):
#     """
#     Dot plot:
#       - y-axis: internal groups (pathway)
#       - x-axis: enriched terms
#       - size: number of overlapping genes (overlap_k)
#       - color: -log10(adjusted p-value)
#     """
#     df = enrich_df.copy()

#     # Categorical encodings for axes
#     path_cats = pd.Categorical(df["pathway"])
#     term_cats = pd.Categorical(df["Term"])

#     df["y"] = path_cats.codes
#     df["x"] = term_cats.codes

#     # Sizes based on overlap_k
#     k = df["overlap_k"].fillna(0).astype(float)
#     if k.max() > 0:
#         k_norm = (k - k.min()) / (k.max() - k.min() + 1e-8)
#     else:
#         k_norm = k
#     size_min, size_max = 30, 300  # points^2
#     df["dot_size"] = size_min + k_norm * (size_max - size_min)

#     # Colors based on -log10(adjP)
#     cvals = df["minus_log10_adjP"]
#     norm = mpl.colors.Normalize(vmin=cvals.min(), vmax=8)
#     cmap = plt.cm.viridis

#     fig, ax = plt.subplots(
#         figsize=(0.22 * len(term_cats.categories) + 4,
#                  0.2 * len(path_cats.categories) + 3)
#     )

#     sc = ax.scatter(
#         df["x"],
#         df["y"],
#         s=df["dot_size"],
#         c=cvals,
#         cmap=cmap,
#         norm=norm,
#         edgecolor="k",
#         linewidth=0.3,
#         alpha=0.9,
#     )

#     ax.set_xticks(np.arange(len(term_cats.categories)))
#     ax.set_xticklabels(term_cats.categories, rotation=90, ha="right")

#     ax.set_yticks(np.arange(len(path_cats.categories)))
#     ax.set_yticklabels(path_cats.categories)

#     ax.set_xlabel("Enriched signature")
#     ax.set_ylabel("Internal group / pathway")

#     ax.set_title(title)

#     cbar = fig.colorbar(sc, ax=ax, pad=0.01)
#     cbar.set_label("-log10(Adjusted P-value)")

#     # Add size legend
#     # Create size legend elements
#     size_legend_values = [5, 10, 20, 40]
#     k_array = np.array(size_legend_values, dtype=float)
#     if k_array.max() > 0:
#         k_norm_legend = (k_array - k_array.min()) / (k_array.max() - k_array.min() + 1e-8)
#     else:
#         k_norm_legend = k_array
#     sizes_legend = size_min + k_norm_legend * (size_max - size_min)
    
#     # Create scatter plot handles for legend
#     legend_handles = [
#         plt.scatter([], [], s=size, c='gray', edgecolor='k', linewidth=0.5, alpha=0.7)
#         for size in sizes_legend
#     ]
    
#     # Add size legend
#     size_legend = ax.legend(
#         legend_handles, 
#         [f'{int(v)} genes' for v in size_legend_values],
#         scatterpoints=1,
#         frameon=False,
#         labelspacing=1.5,
#         title='Overlap size',
#         loc='upper left',
#         bbox_to_anchor=(1.1, 1.0),
#         fontsize=9,
#         title_fontsize=10
#     )
#     ax.add_artist(size_legend)

#     plt.tight_layout()
    
#     # Save the figure before showing
#     plt.savefig(f'{paths.FIGURES_DIR}/replogle/hallmark_enrichment_dotplot.png', dpi=300, bbox_inches='tight')
#     plt.savefig(f'{paths.FIGURES_DIR}/replogle/hallmark_enrichment_dotplot.pdf', dpi=300, bbox_inches='tight')
#     plt.savefig(f'{paths.FIGURES_DIR}/replogle/hallmark_enrichment_dotplot.svg', dpi=300, bbox_inches='tight')
    
#     plt.show()
    
#     print("Figure saved to ../results/hallmark_enrichment_dotplot.png and .pdf")



# enrich = run_group_enrichment_hallmark(
#     pathway_targets=pathway_targets,
#     min_genes_per_group=5,
#     max_genes_per_group=200,
#     hallmark_lib="MSigDB_Hallmark_2020",
#     organism="Human",
#     top_terms_per_group=20,
#     adj_alpha=0.05,   
# )

# plot_hallmark_dotplot(enrich, title="Pathway-level MSigDB_Hallmark_2020 enrichment of downstream targets")

### Fig. 1f: Hallmark enrichment of downstream targets per inferred group

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import gseapy as gp
import time

mpl.rcParams.update({
    'font.size': 10,
    'svg.fonttype': 'none',
    'font.family': 'Arial',
})
mpl.rcParams['axes.titlesize'] = 10
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10


def run_group_enrichment_hallmark(
    pathway_targets: pd.DataFrame,
    min_genes_per_group: int = 5,
    max_genes_per_group: int = 200,
    hallmark_lib: str = "MSigDB_Oncogenic_Signatures",
    organism: str = "Human",
    top_terms_per_group: int = 5,
    adj_alpha: float = 0.05,
):
    df = pathway_targets.copy()

    df = df[df["pathway"] != "None"].copy()

    if "mean_ev" in df.columns:
        df = df.sort_values(["pathway", "mean_ev"], ascending=[True, False])

    path_to_genes = (
        df.groupby("pathway")["target_gene"]
          .apply(lambda s: list(dict.fromkeys(s.astype(str))))
          .to_dict()
    )

    rows = []

    for pw, genes in path_to_genes.items():
        genes = genes[:max_genes_per_group]

        if len(genes) < min_genes_per_group:
            continue

        print(f"Running enrichment for: {pw} ({len(genes)} genes)")
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=[hallmark_lib],
            organism=organism,
            outdir=None,
            cutoff=1.0,
        )
        time.sleep(1)

        res = enr.results.copy()
        if res.empty:
            continue

        res["pathway"] = pw

        def parse_k(s):
            try:
                k, n = s.split("/")
                return int(k)
            except Exception:
                return np.nan

        res["overlap_k"] = res["Overlap"].astype(str).apply(parse_k)
        res["minus_log10_adjP"] = -np.log10(
            res["Adjusted P-value"].clip(lower=1e-300)
        )

        res = res[res["Adjusted P-value"] <= adj_alpha]
        if res.empty:
            continue

        res = res.sort_values("Adjusted P-value").head(top_terms_per_group)

        rows.append(
            res[
                ["pathway", "Term", "Adjusted P-value",
                 "Overlap", "overlap_k", "minus_log10_adjP"]
            ]
        )

    if not rows:
        raise ValueError(
            f"No significant enrichment results at adj_alpha = {adj_alpha}."
        )

    enrich_df = pd.concat(rows, ignore_index=True)
    return enrich_df


def plot_hallmark_dotplot(enrich_df: pd.DataFrame, title: str = ""):
    df = enrich_df.copy()

    path_cats = pd.Categorical(df["pathway"])
    term_cats = pd.Categorical(df["Term"])

    df["y"] = path_cats.codes
    df["x"] = term_cats.codes

    k = df["overlap_k"].fillna(0).astype(float)
    if k.max() > 0:
        k_norm = (k - k.min()) / (k.max() - k.min() + 1e-8)
    else:
        k_norm = k

    size_min, size_max = 30, 300
    df["dot_size"] = size_min + k_norm * (size_max - size_min)

    cvals = df["minus_log10_adjP"]
    norm = mpl.colors.Normalize(vmin=cvals.min(), vmax=8)
    cmap = plt.cm.viridis

    # >>> ONLY CHANGE: make figure wider <<<
    fig, ax = plt.subplots(
        figsize=(0.22 * len(term_cats.categories) + 4,
                 0.2 * len(path_cats.categories) + 3)
    )

    sc = ax.scatter(
        df["x"],
        df["y"],
        s=df["dot_size"],
        c=cvals,
        cmap=cmap,
        norm=norm,
        edgecolor="k",
        linewidth=0.3,
        alpha=0.9,
    )

    ax.set_xticks(np.arange(len(term_cats.categories)))
    ax.set_xticklabels(term_cats.categories, rotation=90, ha="right")

    ax.set_yticks(np.arange(len(path_cats.categories)))
    ax.set_yticklabels(path_cats.categories)

    ax.set_xlabel("Enriched signature")
    ax.set_ylabel("Internal group / pathway")
    ax.set_title(title)

    cbar = fig.colorbar(sc, ax=ax, pad=0.01)
    cbar.set_label("-log10(Adjusted P-value)")

    size_legend_values = [5, 10, 20, 40]
    k_array = np.array(size_legend_values, dtype=float)

    if k_array.max() > 0:
        k_norm_legend = (k_array - k_array.min()) / (
            k_array.max() - k_array.min() + 1e-8
        )
    else:
        k_norm_legend = k_array

    sizes_legend = size_min + k_norm_legend * (size_max - size_min)

    legend_handles = [
        plt.scatter([], [], s=size, c='gray', edgecolor='k',
                    linewidth=0.5, alpha=0.7)
        for size in sizes_legend
    ]

    size_legend = ax.legend(
        legend_handles,
        [f'{int(v)} genes' for v in size_legend_values],
        scatterpoints=1,
        frameon=False,
        labelspacing=1.5,
        title='Overlap size',
        loc='upper left',
        bbox_to_anchor=(1.08, 1.0),
        fontsize=9,
        title_fontsize=10
    )
    ax.add_artist(size_legend)

    plt.tight_layout()

    plt.savefig(f'{paths.FIGURES_DIR}/replogle/hallmark_enrichment_dotplot.png',
                dpi=300, bbox_inches='tight', bbox_extra_artists=(size_legend,))
    plt.savefig(f'{paths.FIGURES_DIR}/replogle/hallmark_enrichment_dotplot.pdf',
                dpi=300, bbox_inches='tight', bbox_extra_artists=(size_legend,))
    plt.savefig(f'{paths.FIGURES_DIR}/replogle/hallmark_enrichment_dotplot.svg',
                dpi=300, bbox_inches='tight', bbox_extra_artists=(size_legend,))


    plt.show()

    print("Figure saved to ../results/hallmark_enrichment_dotplot.png and .pdf")


enrich = run_group_enrichment_hallmark(
    pathway_targets=pathway_targets,
    min_genes_per_group=5,
    max_genes_per_group=200,
    hallmark_lib="MSigDB_Hallmark_2020",
    organism="Human",
    top_terms_per_group=20,
    adj_alpha=0.05,
)

plot_hallmark_dotplot(
    enrich,
    title="Pathway-level MSigDB_Hallmark_2020 enrichment of downstream targets"
)

In [ ]:
z_sub_umap = sub[sub.obs["pert"].isin(clustered_genes)].copy()


In [ ]:
sc.pp.normalize_total(z_sub_umap)
sc.pp.log1p(z_sub_umap)
sc.pp.pca(z_sub_umap, random_state=0)

sc.pp.neighbors(z_sub_umap, random_state=0)
sc.tl.umap(z_sub_umap, random_state=0)


In [ ]:
z_sub_umap.obs["true_pathway"]   = z_sub_umap.obs["pert"].map(gene_to_true)


### Fig. 1d: UMAPs of the expression space and the SHERLOCK latent space, by pathway

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_hex
import matplotlib.pyplot as plt
import matplotlib as mpl
# Update your existing rcParams
mpl.rcParams.update({
    'font.size': 8,
    'svg.fonttype': 'none',
    'font.family': 'Arial',
    'figure.figsize': (3, 2.25)  # Set default figure size to 6×4 inches
})
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['xtick.labelsize'] = 8  # Colorbar tick label size 
mpl.rcParams['ytick.labelsize'] = 8  # Colorbar tick label size

# --- your existing setup ---
results_df = results['merged']
clustered_genes = results_df["gene"].unique()

z_sub = sub[sub.obs["pert"].isin(clustered_genes)].copy()

gene_to_true   = dict(zip(results_df["gene"], results_df["pathway"]))
gene_to_mapped = dict(zip(results_df["gene"], results_df["mapped_pathway"]))

z_sub.obs["true_pathway"]   = z_sub.obs["pert"].map(gene_to_true)
z_sub.obs["mapped_pathway"] = z_sub.obs["pert"].map(gene_to_mapped)

# Drop NA mapped rows
z_sub = z_sub[z_sub.obs["mapped_pathway"].notna() & (z_sub.obs["mapped_pathway"] != "NA")].copy()

# --- NEW: enforce identical palettes for both columns ---
# 1) Make both columns categorical with the SAME category set & order
true_vals   = pd.Index(z_sub.obs["true_pathway"].dropna().unique().astype(str))
mapped_vals = pd.Index(z_sub.obs["mapped_pathway"].dropna().unique().astype(str))
all_cats = pd.Index(np.unique(true_vals.union(mapped_vals)))  # sorted union

cat_dtype = pd.api.types.CategoricalDtype(categories=all_cats, ordered=False)
z_sub.obs["true_pathway"]   = z_sub.obs["true_pathway"].astype("string").astype(cat_dtype)
z_sub.obs["mapped_pathway"] = z_sub.obs["mapped_pathway"].astype("string").astype(cat_dtype)

# 2) Build a single color palette over all categories
cmap = plt.cm.get_cmap("tab20", len(all_cats))  # any qualitative map works
shared_palette = {cat: to_hex(cmap(i)) for i, cat in enumerate(all_cats)}

# 3) Write per-column color lists in each column's category order
true_order   = list(z_sub.obs["true_pathway"].cat.categories)
mapped_order = list(z_sub.obs["mapped_pathway"].cat.categories)

z_sub.uns["true_pathway_colors"]   = [shared_palette[c] for c in true_order]
z_sub.uns["mapped_pathway_colors"] = [shared_palette[c] for c in mapped_order]

# --- Plot (colors now matched across panels) ---

# Alternative: Save manually with more control
fig = sc.pl.umap(
    z_sub,
    color=["true_pathway"],
    wspace=0.4,
    show=False,
    return_fig=True,
    title="Pathways",
    s= 5,
    frameon=False,
)


fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison.pdf', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison.svg', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to ../results/umap_pathway_comparison.png and .pdf")


In [ ]:
# Alternative: Save manually with more control
fig = sc.pl.umap(
    z_sub,
    color=["pert"],
    wspace=0.4,
    show=False,
    return_fig=True,
    title="Perturbations",
    s= 5,
    frameon=False,
)

fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison_pert.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison_pert.pdf', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pathway_comparison_pert.svg', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to ../results/umap_pathway_comparison.png and .pdf")


In [ ]:
z_sub_umap.uns["true_pathway_colors"]   = [shared_palette[c] for c in true_order]

In [ ]:
fig = sc.pl.umap(
    z_sub_umap,
    color=['true_pathway'],
    show=False,
    wspace=0.4,
    return_fig=True,
    frameon=False,
    title="Pathways",
    s= 5,
)

fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_true_pathway.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_true_pathway.pdf', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_true_pathway.svg', dpi=300, bbox_inches='tight')
plt.show()

print('Figure saved to ../results/umap_true_pathway.png/.pdf/.svg')


### Fig. 1d: Leiden clusters of the expression space

In [ ]:
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Perform clustering on UMAP space ---
# Run leiden clustering on the UMAP neighbors
sc.tl.leiden(z_sub_umap, resolution=0.5, key_added='leiden_clusters', random_state=0)

sc.pl.umap(z_sub_umap, color ='leiden_clusters' )
# --- Extract cluster labels and true pathway labels ---
cluster_labels = z_sub_umap.obs['leiden_clusters'].astype(str).values
true_pathway_labels = z_sub_umap.obs['true_pathway'].astype(str).values

# Remove any NA values
mask = (true_pathway_labels != 'nan') & (true_pathway_labels != 'None')
cluster_labels = cluster_labels[mask]
true_pathway_labels = true_pathway_labels[mask]

# --- Calculate ARI ---
ari = adjusted_rand_score(true_pathway_labels, cluster_labels)
print(f"ARI (true_pathway vs UMAP leiden clusters): {ari:.4f}")

# --- Confusion matrix using crosstab ---
# Row normalize: each row sums to 1 (recall per true pathway)
cm_df = pd.crosstab(
    true_pathway_labels, 
    cluster_labels, 
    normalize='index'
).fillna(0)

# Get labels for axes
true_labels = sorted(cm_df.index.astype(str))
cluster_ids = sorted(cm_df.columns.astype(str))

# --- Plot confusion matrix ---
fig, ax = plt.subplots(figsize=(len(cluster_ids) * 0.6 + 2,
                                len(true_labels) * 0.5 + 2))
sns.heatmap(cm_df, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Fraction of true pathway'})
ax.set_xlabel('UMAP leiden cluster', fontsize=14)
ax.set_ylabel('True pathway', fontsize=14)
ax.set_title(f'UMAP clustering vs true pathways (ARI = {ari:.3f})', fontsize=14)
plt.tight_layout()

# --- Save figures ---
plt.savefig(f'{paths.FIGURES_DIR}/replogle/umap_leiden_vs_true_pathway_cm.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{paths.FIGURES_DIR}/replogle/umap_leiden_vs_true_pathway_cm.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{paths.FIGURES_DIR}/replogle/umap_leiden_vs_true_pathway_cm.svg', dpi=300, bbox_inches='tight')
plt.show()

print("Saved confusion matrix to ../results/umap_leiden_vs_true_pathway_cm.png/.pdf/.svg")
print(f"\nSummary:")
print(f"  - True pathways: {len(true_labels)}")
print(f"  - UMAP clusters: {len(cluster_ids)}")
print(f"  - ARI: {ari:.4f}")


In [ ]:
z_sub_umap.obs["mapped_pathway"] = z_sub_umap.obs["pert"].map(gene_to_mapped)

In [ ]:
# --- Extract cluster labels and true pathway labels ---
cluster_labels = z_sub_umap.obs['mapped_pathway'].astype(str).values
true_pathway_labels = z_sub_umap.obs['true_pathway'].astype(str).values

# Remove any NA values
mask = (true_pathway_labels != 'nan') & (true_pathway_labels != 'None')
cluster_labels = cluster_labels[mask]
true_pathway_labels = true_pathway_labels[mask]

# --- Calculate ARI ---
ari = adjusted_rand_score(true_pathway_labels, cluster_labels)
print(f"ARI (true_pathway vs UMAP leiden clusters): {ari:.4f}")

# --- Confusion matrix using crosstab ---
# Row normalize: each row sums to 1 (recall per true pathway)
cm_df = pd.crosstab(
    true_pathway_labels, 
    cluster_labels, 
    normalize='index'
).fillna(0)

# Get labels for axes
true_labels = sorted(cm_df.index.astype(str))
cluster_ids = sorted(cm_df.columns.astype(str))

# --- Plot confusion matrix ---
fig, ax = plt.subplots(figsize=(len(cluster_ids) * 0.6 + 2,
                                len(true_labels) * 0.5 + 2))
sns.heatmap(cm_df, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Fraction of true pathway'})
ax.set_xlabel('Mapped Pathway', fontsize=14)
ax.set_ylabel('True pathway', fontsize=14)
ax.set_title(f'UMAP clustering vs true pathways (ARI = {ari:.3f})', fontsize=14)
plt.tight_layout()

# --- Save figures ---
plt.savefig(f'{paths.FIGURES_DIR}/replogle/umap_leiden_vs_true_pathway_cm.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{paths.FIGURES_DIR}/replogle/umap_leiden_vs_true_pathway_cm.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{paths.FIGURES_DIR}/replogle/umap_leiden_vs_true_pathway_cm.svg', dpi=300, bbox_inches='tight')
plt.show()

print("Saved confusion matrix to ../results/umap_leiden_vs_true_pathway_cm.png/.pdf/.svg")
print(f"\nSummary:")
print(f"  - True pathways: {len(true_labels)}")
print(f"  - UMAP clusters: {len(cluster_ids)}")
print(f"  - ARI: {ari:.4f}")


In [ ]:
fig = sc.pl.umap(
    z_sub_umap,
    color=['leiden_clusters'],
    show=False,
    wspace=0.4,
    return_fig=True,
    frameon=False,
    title="Clusters",
    s= 5,
)

fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_clustered_true_pathway.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_clustered_true_pathway.pdf', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_clustered_true_pathway.svg', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
fig = sc.pl.umap(
    z_sub_umap,
    color=['pert'],
    show=False,
    wspace=0.4,
    return_fig=True,
    frameon=False,
    title="Pathways",
    s= 5,
)

fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pert.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pert.pdf', dpi=300, bbox_inches='tight')
fig.savefig(f'{paths.FIGURES_DIR}/replogle/umap_pert.svg', dpi=300, bbox_inches='tight')
plt.show()

print('Figure saved to ../results/umap_pert.png/.pdf/.svg')


In [ ]:
data.obs["true_pathway"]   = data.obs["pert"].map(gene_to_true)
data.obs["mapped_pathway"] = data.obs["pert"].map(gene_to_mapped)

In [ ]:
mpl.rcParams.update({
    'font.size': 8,
    'svg.fonttype': 'none'
})
mpl.rcParams['axes.titlesize'] = 8
mpl.rcParams['xtick.labelsize'] = 2  # Colorbar tick label size 
mpl.rcParams['ytick.labelsize'] = 2  # Colorbar tick label size
mpl.rcParams['xtick.labelsize'] = 1  # x-axis tick label size
mpl.rcParams['ytick.labelsize'] = 1  # y-axis tick label size

from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib import gridspec
from matplotlib.patches import Patch


def plot_zcorr_colored(adata, uns_key='results', show_pert_labels=True, palette=shared_palette):
    """
    Plot correlation matrix with two color-bar trackers (top only):
      - top bar row 1 : true_pathway   (from shared_palette)
      - top bar row 2 : mapped_pathway (from shared_palette)
    """

    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']
    pert_names = adata.uns[uns_key]['rho_perts']

    link = linkage(Sigma_P_posterior, method="average", metric="euclidean")
    order = leaves_list(link)

    SigmaP_ord = Sigma_P_posterior[order][:, order]
    pert_names_ord = pert_names[order]

    # Map each perturbation -> true_pathway / mapped_pathway color
    p_key = slk.configs.get_config('pert_key')
    pert_to_true   = adata.obs.groupby(p_key)['true_pathway'].first().to_dict()
    pert_to_mapped = adata.obs.groupby(p_key)['mapped_pathway'].first().to_dict()

    fallback = '#cccccc'
    true_colors   = [palette.get(str(pert_to_true.get(n,   '')), fallback) for n in pert_names_ord]
    mapped_colors = [palette.get(str(pert_to_mapped.get(n, '')), fallback) for n in pert_names_ord]

    n_perts = len(pert_names_ord)
    bar_thick = 35  # thicker bars

    # Plot heatmap
    fig = plt.figure(figsize=(n_perts * 0.01, n_perts * 0.01))
    ax0 = fig.add_subplot(111)

    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax0,
                xticklabels=pert_names_ord if show_pert_labels else False,
                yticklabels=pert_names_ord if show_pert_labels else False,
                cbar_kws={'label': 'Correlation', 'shrink': 0.4, 'pad': 0.1})
    ax0.set_xlabel("Perturbation", fontsize=20)
    ax0.set_ylabel("Perturbation", fontsize=20)
    ax0.tick_params(axis='both', which='both', length=0)

    if show_pert_labels:
        ax0.set_xticklabels(ax0.get_xticklabels(), fontsize=1)
        ax0.set_yticklabels(ax0.get_yticklabels(), fontsize=1)

    # Colorbar label font size
    cbar = ax0.collections[0].colorbar
    cbar.set_label('Correlation', fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    # Two stacked color-bar trackers (right side only)
    for i, (tc, mc) in enumerate(zip(true_colors, mapped_colors)):
        # Col 1: true_pathway
        ax0.add_patch(plt.Rectangle((n_perts + 5,                  i), bar_thick, 1, color=tc,
                                    transform=ax0.transData, clip_on=False))
        # Col 2: mapped_pathway (gap of 3 units between bars)
        ax0.add_patch(plt.Rectangle((n_perts + 5 + bar_thick + 3,  i), bar_thick, 1, color=mc,
                                    transform=ax0.transData, clip_on=False))

    # # Corner annotation: label left/right bars
    # ax0.text(1.3, 0.99, 'Left: True pathways\nRight: Sherlock mapped pathways',
    #      ha='left', va='top', fontsize=11, transform=ax0.transAxes,
    #      bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7, edgecolor='none'))

    # Legend
    shown_pathways = sorted({p for p in
                              list(pert_to_true.values()) + list(pert_to_mapped.values())
                              if p and str(p) != 'nan' and p in palette})
    legend_elements = [Patch(facecolor=palette[p], label=p) for p in shown_pathways]
    fig.legend(
        handles=legend_elements,
        loc='center left',
        bbox_to_anchor=(1, 0.5),
        title='Pathway',
        fontsize=14,
        title_fontsize=16,
        frameon=False,
    )

    plt.tight_layout()
    plt.savefig(f'{paths.FIGURES_DIR}/replogle/benchmark_correlation_color.svg', dpi=300, bbox_inches='tight')
    plt.savefig(f'{paths.FIGURES_DIR}/replogle/benchmark_correlation_color.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{paths.FIGURES_DIR}/replogle/benchmark_correlation_color.pdf', dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_colored(data)

### Fig. 1e: learned perturbation correlation with reference and inferred pathways

In [ ]:
def plot_zcorr_colored(adata, uns_key='results', palette=shared_palette):
    from scipy.cluster.hierarchy import dendrogram

    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    pert_names        = adata.uns[uns_key]['rho_perts']

    link  = linkage(Sigma_P_posterior, method='average', metric='euclidean')
    order = leaves_list(link)

    SigmaP_ord     = Sigma_P_posterior[order][:, order]
    pert_names_ord = pert_names[order]

    p_key          = slk.configs.get_config('pert_key')
    pert_to_true   = adata.obs.groupby(p_key)['true_pathway'].first().to_dict()
    pert_to_mapped = adata.obs.groupby(p_key)['mapped_pathway'].first().to_dict()

    fallback      = '#cccccc'
    true_colors   = [palette.get(str(pert_to_true.get(n,   '')), fallback) for n in pert_names_ord]
    mapped_colors = [palette.get(str(pert_to_mapped.get(n, '')), fallback) for n in pert_names_ord]

    n    = len(pert_names_ord)
    base = max(2.5, n * 0.0025)

    # layout: heatmap | gap | true_strip | gap | mapped_strip | gap | dendrogram | colorbar
    fig = plt.figure(figsize=(base + 2, base))
    print(base)
    gs  = gridspec.GridSpec(
        1, 8,
        width_ratios=[1, 0.02, 0.075, 0.05, 0.075, 0.05, 0.30, 0.04],
        wspace=0.01,
    )
    ax_heat   = fig.add_subplot(gs[0])
    ax_true   = fig.add_subplot(gs[2])
    ax_mapped = fig.add_subplot(gs[4])
    ax_dend   = fig.add_subplot(gs[6])
    ax_cbar   = fig.add_subplot(gs[7])

    # Heatmap
    im = ax_heat.imshow(SigmaP_ord, cmap='coolwarm', aspect='auto',
                        vmin=-1, vmax=1, interpolation='nearest')
    ax_heat.set_xticks([])
    ax_heat.set_yticks([])
    ax_heat.set_xlabel('Perturbation', fontsize=10)
    ax_heat.set_ylabel('Perturbation', fontsize=10)

    # True pathway strip
    ax_true.imshow(np.array([[mpl.colors.to_rgb(c)] for c in true_colors]),
                   aspect='auto', interpolation='nearest')
    ax_true.set_xticks([])
    ax_true.set_yticks([])
    ax_true.set_title('True', fontsize=7)

    # Mapped pathway strip
    ax_mapped.imshow(np.array([[mpl.colors.to_rgb(c)] for c in mapped_colors]),
                     aspect='auto', interpolation='nearest')
    ax_mapped.set_xticks([])
    ax_mapped.set_yticks([])
    ax_mapped.set_title('Mapped', fontsize=7)

    # Build node colors: leaf branches colored by mapped pathway,
    # internal branches colored only if entire subtree is one pathway (else gray)
    gray = '#cccccc'
    leaf_colors_orig = [palette.get(str(pert_to_mapped.get(nm, '')), gray) for nm in pert_names]
    node_color = {i: leaf_colors_orig[i] for i in range(n)}
    for i, row in enumerate(link):
        left, right = int(row[0]), int(row[1])
        cl = node_color.get(left,  gray)
        cr = node_color.get(right, gray)
        node_color[n + i] = cl if cl == cr else gray

    # Dendrogram
    dendrogram(link, ax=ax_dend, orientation='right', no_labels=True,
               color_threshold=0,
               link_color_func=lambda k: node_color.get(k, gray))
    for coll in ax_dend.collections:
        coll.set_linewidth(0.45) 
    ax_dend.set_xticks([])
    ax_dend.set_yticks([])
    ax_dend.invert_yaxis()
    for spine in ax_dend.spines.values():
        spine.set_visible(False)

    # Colorbar
    cbar = fig.colorbar(im, cax=ax_cbar)
    cbar.set_label('Correlation', fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    plt.savefig(f'{paths.FIGURES_DIR}/replogle/benchmark_correlation_color.svg', dpi=300, bbox_inches='tight')
    plt.savefig(f'{paths.FIGURES_DIR}/replogle/benchmark_correlation_color.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{paths.FIGURES_DIR}/replogle/benchmark_correlation_color.pdf', dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_colored(data)

## Replogle: benchmarking

From [`scripts/replogle/2_benchmarking.ipynb`](../scripts/replogle/2_benchmarking.ipynb).

Loads the five saved runs of each method from `results/benchmarking/models/`; missing runs are trained. Needs the `benchmarks` extra of SHERLOCK.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import torch
from matplotlib.colors import to_hex
from scipy.optimize import linear_sum_assignment

import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib as mpl

# ── Journal figure style ────────────────────────────────────────────────
RC = {
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':             8,
    'axes.labelsize':        8,
    'axes.titlesize':        9,
    'xtick.labelsize':       7,
    'ytick.labelsize':       7,
    'legend.fontsize':       6,
    'legend.title_fontsize': 7,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.linewidth':        0.75,
    'xtick.major.width':     0.75,
    'ytick.major.width':     0.75,
    'xtick.major.size':      3,
    'ytick.major.size':      3,
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
    'svg.fonttype': 'none',
}
mpl.rcParams.update(RC)

In [ ]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_RUNS      = 5
NUM_EPOCHS  = 200
VALIDATE_EVERY = 50
BATCH_SIZE  = 4096
LATENT_DIM  = 16
PATIENCE    = 20
RESULTS_DIR = f"{paths.RESULTS_DIR}/benchmarking"
N_CLUSTERS  = 8

# Metrics tracked for scoring and box plots
METRIC_COLS = [
    'cf_ate_pearson_r',
    'cluster_f1',
    'cluster_ari',
    'cluster_ami',
    'cluster_silhouette',
    'cluster_linear_probe',
    'cluster_knn_purity',
]

# (model_key, display_name, method-specific kwargs)
# sherlock (vae) kwargs mirror the tuned hyperparameters in "demo copy.ipynb"
# (cov_lambda / H_lambda / gate_row_repulsion_lambda disabled, ce_lambda raised to 20)
METHODS = [
    ("vae",           "sherlock",      {
        "l0_lambda": 1e-1,
        "ce_lambda": 20.,
        "n_epochs_l0_warmup": 300,
    },  500),
    ("svae",          "sVAE",          {"sparse_mask_penalty": 10.0},  NUM_EPOCHS),
    ("cvae",          "cVAE",          {},  NUM_EPOCHS),
    ("contrastivevi", "contrastiveVI", {},  NUM_EPOCHS),
    ("scgen",         "scgen",         {},  NUM_EPOCHS),
]

####TODO CHECK THIS#####
METHODS = [
    ("vae",           "sherlock",      {
        "l0_lambda": 1e-1,
        "ce_lambda": 20.,
        "n_epochs_l0_warmup": 300,
    }, 500),
    ("svae",          "sVAE",          {
        "sparse_mask_penalty": 4.366473592979633,
        "beta": 0.614252574393396,
        "n_hidden": 251,
        "n_layers": 3,
        "dropout_rate": 0.28184968246925673,
        "lr": 0.00331348361550895,
    }, NUM_EPOCHS),
    ("cvae",          "cVAE",          {
        "beta": 3.2226582291306802,
        "n_hidden": 67,
        "n_layers": 1,
        "dropout_rate": 0.29147206450401036,
        "lr": 0.006205861500406846,
    }, NUM_EPOCHS),
    ("contrastivevi", "contrastiveVI", {
        "wasserstein_penalty": 4.399492045282824,
        "n_hidden": 94,
        "n_layers": 2,
        "dropout_rate": 0.2902495045654139,
        "lr": 0.0005218479707033748,
    }, NUM_EPOCHS),
    ("scgen",         "scgen",         {}, NUM_EPOCHS),
]

os.makedirs(f"{RESULTS_DIR}/models",   exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures",  exist_ok=True)

print(f"Device: {DEVICE}  Results: {RESULTS_DIR}")

## Data loading and ground-truth ATE

In [ ]:
data = slk.datasets.replogle()

slk.pp.treat_effect(
    data,
    label_col='pert',
    control_label='NTC',
    method='perturbseq',
    inplace=True,
    key='treat_effect',
)
treat_effect_adata = data.uns['treat_effect']
print(f"Cells: {data.n_obs}  Genes: {data.n_vars}  Perturbations: {data.obs['pert'].nunique()}")

## Pathway annotations for clustering evaluation

In [ ]:
pathways = data.uns['pathways']

all_genes_filtered = [
    gene
    for genes in pathways.values()
    for gene in genes
    if gene != 'LSM5'
]

gene_to_pathway = {
    gene: pathway
    for pathway, genes in pathways.items()
    for gene in genes
    if gene != 'LSM5'
}
pathway_df_indexed = pd.DataFrame.from_dict(
    gene_to_pathway, orient='index', columns=['pathway']
)
pathway_df_indexed.index.name = 'gene'

print(f"Pathway genes: {len(all_genes_filtered)}  Pathways: {len(pathways)}")

## Benchmark loop

Each method is trained `N_RUNS` times independently. The best-scoring run
per method is saved to `results/models/`. Metrics are computed after training
using the proper counterfactual ATE protocol; the clustering cut is chosen
automatically via the elbow method.

In [ ]:
records         = []
best_run_out    = {}   # method → out dict of best single run
best_run_model  = {}   # method → model of best single run
best_run_score  = {}   # method → avg metric score of best single run
best_run_path   = {}   # method → run_x.pth of best single run

for model_key, display_name, extra_kwargs, EPOCHS, in METHODS:
    print(f"\n{'='*60}\n{display_name}  ({N_RUNS} runs)\n{'='*60}")
    best_run_score[display_name] = -np.inf

    # Each method gets its own folder; individual runs are checkpointed there
    # so a re-executed notebook reloads finished runs instead of retraining.
    model_dir = f"{RESULTS_DIR}/models/{display_name}"
    os.makedirs(model_dir, exist_ok=True)

    for run in range(N_RUNS):
        print(f"\n  -- {display_name} run {run + 1}/{N_RUNS} --")

        run_path = f"{model_dir}/run_{run}.pth"

        if os.path.exists(run_path):
            print(f"    Found checkpoint → {run_path}, loading instead of retraining")
            model = slk.tl.load_results(run_path).to(DEVICE)
            out = {"model": model, "param_store": None}
        else:
            out = slk.tl.run(
                data,
                model=model_key,
                batch_size=BATCH_SIZE,
                num_epochs=EPOCHS,
                device=DEVICE,
                latent_dim=LATENT_DIM,
                patience=PATIENCE,
                validate_every=VALIDATE_EVERY,
                seed=run,
                **extra_kwargs,
            )
            model = out['model']
            slk.tl.save_results(out, run_path)
            print(f"    Saved → {run_path}")

        metrics = slk.tl.evaluate_model(
            model,
            data,
            treat_effect_adata,
            pathway_df_indexed,
            all_genes_filtered,
            n_clusters=N_CLUSTERS,
            device=DEVICE,
            use_rho=False,
        )

        # Track best run for this method
        run_avg = float(metrics['cf_ate_pearson_r'])
        if run_avg > best_run_score[display_name]:
            best_run_score[display_name] = run_avg
            best_run_out[display_name]   = out
            best_run_model[display_name] = model
            best_run_path[display_name]  = run_path

        metrics['method'] = display_name
        metrics['run']    = run
        records.append(metrics)

        for k, v in metrics.items():
            if k not in ('method', 'run'):
                val_str = f"{v:.4f}" if np.isfinite(float(v)) else "NaN"
                print(f"    {k:30s} = {val_str}")

    # Save best run model as an exact copy of its own run_x.pth. Copying (vs.
    # re-saving from memory) guarantees best.pth carries the best run's
    # state_dict AND its Pyro param store (sigma_fac, …). The store is not in
    # state_dict, and the *global* Pyro store reflects whichever run ran last,
    # not necessarily the best — so we never rely on it.
    import shutil
    save_path = f"{model_dir}/best.pth"
    shutil.copyfile(best_run_path[display_name], save_path)
    print(f"\n  Saved best {display_name} model → {save_path} "
          f"(copied from {os.path.basename(best_run_path[display_name])})")

results_df = pd.DataFrame(records)
print("\nDone.")

## Results summary

In [ ]:
summary = (
    results_df
    .groupby('method')[METRIC_COLS]
    .agg(['mean', 'std'])
    .round(4)
)
# Preserve METHODS order
method_order = [m[1] for m in METHODS]
summary = summary.reindex([m for m in method_order if m in summary.index])
print(summary.to_string())

## Box plots

### Fig. 1g: benchmark of SHERLOCK, cVAE, sVAE+, contrastiveVI and scGen

In [ ]:
# Toggle significance brackets (SHERLOCK vs each baseline, per metric column).
# Welch's t-test on the per-run values, Benjamini-Hochberg FDR across the whole
# family, drawn ggsignif-style as ns / * / ** / *** (same as norman_figure Panel A).
SIG = True

CLR_SHERLOCK    = '#2166ac'
CLR_GEARS       = '#d73027'

METHOD_PALETTE = {
    'SHERLOCK':      CLR_SHERLOCK,
    'cVAE':          '#33a02c',
    'sVAE':          '#ff7f00',
    'contrastiveVI': '#6a3d9a',
    'GEARS':         CLR_GEARS,
}

label_map = {
    'cf_ate_pearson_r':   'ATE',
    'cluster_f1':         'F1',
    'cluster_ari':        'ARI',
    'cluster_ami':        'AMI',
    'cluster_silhouette': 'Silhouette',
    # 'cluster_linear_probe': 'Linear\nProbe',
    # 'cluster_knn_purity': 'kNN\nPurity',
}

METRIC_COLS_PLOT = [
    m for m in METRIC_COLS
    if m not in ['cluster_linear_probe', 'cluster_knn_purity']
]

# Display names matched to METHOD_PALETTE keys (sherlock -> SHERLOCK)
method_display = {
    'sherlock':      'SHERLOCK',
}
method_order = [method_display.get(m[1], m[1]) for m in METHODS]

df_melted = results_df.melt(
    id_vars='method', value_vars=METRIC_COLS_PLOT, var_name='metric', value_name='value'
)
df_melted['method'] = df_melted['method'].map(lambda m: method_display.get(m, m))
df_melted['metric_label'] = pd.Categorical(
    df_melted['metric'].map(label_map),
    categories=[label_map[m] for m in METRIC_COLS_PLOT],
    ordered=True,
)
df_melted['method'] = pd.Categorical(
    df_melted['method'], categories=method_order, ordered=True
)
df_melted['value'] = df_melted['value'].clip(lower=0)

# scgen has no ATE result to show
df_melted = df_melted[~((df_melted['method'] == 'scgen') & (df_melted['metric'] == 'cf_ate_pearson_r'))]

# Fall back to gray for any method not in METHOD_PALETTE (scgen is colored red)
palette = {m: METHOD_PALETTE.get(m, '#999999') for m in method_order}
palette['scgen'] = CLR_GEARS

# Taller when SIG is on so the stacked brackets have headroom above the boxes.
fig, ax = plt.subplots(figsize=(6.5*0.75, (2.7 if SIG else 2)*0.75))

# NOTE: stripplot's dodge always spans a fixed width of 0.8 (it has no `width`
# kwarg), so the boxplot must also dodge over width=0.8 for box/point centers
# to line up -- otherwise outer hue levels (e.g. scgen) drift off to the side.
# `gap` shrinks the box's visual thickness without moving its dodge position.
sns.boxplot(
    data=df_melted,
    x='metric_label',
    y='value',
    hue='method',
    hue_order=method_order,
    palette=palette,
    width=0.8,
    gap=0.25,
    linewidth=0.6,
    showfliers=False,
    boxprops=dict(alpha=0.55),
    ax=ax,
)

sns.stripplot(
    data=df_melted,
    x='metric_label',
    y='value',
    hue='method',
    hue_order=method_order,
    palette=palette,
    dodge=True,
    jitter=0.1,
    size=3,
    linewidth=0.3,
    edgecolor='white',
    legend=False,
    ax=ax,
)

ax.set_xlabel('')
ax.set_ylabel('Score')
ax.legend(title='Method', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_ylim(0, 1.0)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# Separator between ATE and clustering metrics
ax.axvline(0.5, color='lightgray', linewidth=1.0, linestyle=':')

# ── Significance: SHERLOCK vs every other method, per metric column ────────
# Brackets are drawn above the y=1.0 axis (clip_on=False) so the score axis
# stays a clean 0-1; savefig uses bbox_inches='tight' so they aren't clipped.
if SIG:
    from scipy import stats as _stats
    _sig_metrics = METRIC_COLS_PLOT                        # raw metric names, in column order
    _sig_others  = [m for m in method_order if m != 'SHERLOCK']

    def _mvals(_mk, _meth):
        return df_melted[(df_melted['metric'] == _mk) &
                         (df_melted['method'] == _meth)]['value'].dropna().to_numpy()

    _comps = []                                            # [col_i, other_method, p]
    for _i, _mk in enumerate(_sig_metrics):
        _s = _mvals(_mk, 'SHERLOCK')
        for _o in _sig_others:
            _ov = _mvals(_mk, _o)
            if len(_s) < 2 or len(_ov) < 2:
                continue
            _comps.append([_i, _o, _stats.ttest_ind(_s, _ov, equal_var=False).pvalue])

    # Benjamini-Hochberg across the whole family
    _pv = np.array([c[2] for c in _comps])
    _ord = np.argsort(_pv); _mtot = len(_pv); _adj = np.empty(_mtot); _prev = 1.0
    for _k in range(_mtot - 1, -1, -1):
        _ix = _ord[_k]; _prev = min(_prev, _pv[_ix] * _mtot / (_k + 1)); _adj[_ix] = min(_prev, 1.0)
    for _c, _pa in zip(_comps, _adj):
        _c.append(_pa)

    def _stars(_p):
        return '***' if _p < 1e-3 else '**' if _p < 1e-2 else '*' if _p < 0.05 else 'ns'

    # dodge geometry (boxplot width=0.8, one box per method)
    _n_hue, _box_w = len(method_order), 0.8
    def _xpos(_i, _meth):
        _j = method_order.index(_meth)
        return _i - _box_w / 2 + _box_w / (2 * _n_hue) + _j * (_box_w / _n_hue)

    _y0   = 1.03                                           # first bracket just above the y=1.0 axis
    _step = 0.075                                          # gap between stacked brackets
    _tick = 0.012                                          # bracket end-tick height
    _lbl_h = 0.028                                         # ink height reserved for a star label
    # Asterisk glyphs sit high in the text box; 'ns' letters sit on the baseline.
    # These per-label nudges land the ink at the same height above every bracket
    # line so all labels read as equal-height regardless of glyph.
    _lbl_dy = {'ns': 0.006}
    _ytop = _y0
    for _i in range(len(_sig_metrics)):
        _col = sorted([c for c in _comps if c[0] == _i],
                      key=lambda c: method_order.index(c[1]))   # nearest method lowest
        for _lvl, (_, _o, _p, _pa) in enumerate(_col):
            _x1, _x2 = _xpos(_i, 'SHERLOCK'), _xpos(_i, _o)
            _y = _y0 + _lvl * _step
            ax.plot([_x1, _x1, _x2, _x2], [_y, _y + _tick, _y + _tick, _y],
                    lw=0.5, c='0.35', clip_on=False)
            _lbl = _stars(_pa)
            ax.text((_x1 + _x2) / 2, _y + _tick + _lbl_dy.get(_lbl, -0.014), _lbl,
                    ha='center', va='bottom', fontsize=5, color='0.2', clip_on=False)
            _ytop = max(_ytop, _y + _tick + _lbl_h)

    # Title above the tallest bracket (axis is 0-1, so a data value == axes frac).
    ax.set_title('Replogle Benchmarking Results', y=_ytop + 0.04)
else:
    ax.set_title('Replogle Benchmarking Results', pad=6)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/figures/benchmark_boxplot.svg', bbox_inches='tight')
plt.savefig(f'{RESULTS_DIR}/figures/benchmark_boxplot.pdf', bbox_inches='tight')
plt.savefig(f'{RESULTS_DIR}/figures/benchmark_boxplot.png', bbox_inches='tight')
plt.show()